In [1]:
# load_data.py
import pandas as pd
from sqlalchemy import create_engine
from config import CONFIG


In [2]:
engine = create_engine(CONFIG["db_url"])


In [3]:

def load_raw_table(engine, schema, table):
    query = f'SELECT * FROM "{schema}"."{table}"'
    df    = pd.read_sql(query, engine)
    print(f"Loaded : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
    print(f"Memory : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    return df


In [4]:
df = load_raw_table(engine, CONFIG["schema"], CONFIG["raw_table"])

Loaded : 101,766 rows  x  50 columns
Memory : 206.0 MB


In [5]:
# Profiling
import pandas as pd
import numpy as np
from config import CONFIG

In [6]:
def profile_structure(df):
    profile = pd.DataFrame({
        "dtype": df.dtypes,
        "non_null": df.notna().sum(),
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().sum() / len(df) * 100).round(2),
        "unique": df.nunique(),
        "sample_value": df.apply(
            lambda col: col.dropna().iloc[0] if col.notna().any() else "ALL NULL"
        )
    })

    profile = (
        profile
        .reset_index()
        .rename(columns={"index": "column"})
        .sort_values("null_pct", ascending=False)
    )

    return profile

In [7]:
structure_raw = profile_structure(df)

In [8]:
print(structure_raw.to_string(index=False))

                  column   dtype  non_null  null_count  null_pct  unique             sample_value
            encounter_id   int64    101766           0       0.0  101766                  2278392
             patient_nbr   int64    101766           0       0.0   71518                  8222157
                    race     str    101766           0       0.0       6                Caucasian
                  gender     str    101766           0       0.0       3                   Female
                     age     str    101766           0       0.0      10                   [0-10)
                  weight     str    101766           0       0.0      10                        ?
       admission_type_id float64    101765           1       0.0       8                      6.0
discharge_disposition_id   int64    101766           0       0.0      26                       25
     admission_source_id   int64    101766           0       0.0      17                        1
        time_in_hosp

##  Structural Profile

**Purpose:**
Capture the shape, data types, null counts, unique value counts, and a sample
value per column before any transformation. This is the baseline — every
cleaning decision references back to this output.

**Dataset Overview:**
- Total rows    : 101,766 encounters
- Total columns : 50 features
- Primary key   : encounter_id (101,766 unique — confirms no duplicates at this stage)
- Unique patients: 71,518 (confirms multiple encounters per patient exist)

**Key Observations from this Profile:**

1. **No nulls visible yet** — this is expected at this stage. Missing values in
   this dataset are encoded as the string "?" rather than NULL. Columns like
   weight, payer_code, diag_2, and diag_3 show null_count = 0 here but will
   reveal true missingness after sentinel replacement in the cleaning phase.

2. **Sentinel "?" already visible in sample_value** — weight, payer_code,
   diag_2, and diag_3 all show "?" as their sample value, confirming that
   sentinel detection and replacement is a critical first cleaning step.

3. **admission_type_id is the only column with 1 null** — this is a genuine
   NULL in the raw data, not a sentinel value. It will be handled in the
   missing value treatment step.

4. **ID columns stored as float/int** — admission_type_id is float64 and
   discharge_disposition_id is int64. These are categorical lookup codes,
   not quantities. They will be cast to string in the cleaning phase to
   prevent accidental arithmetic.

5. **Medication columns all show unique = 2 or 4** — confirming they contain
   only the expected controlled vocabulary values (No, Steady, Up, Down).
   examide and citoglipton show unique = 1, meaning they contain only "No"
   across all 101,766 rows — these columns carry no analytical value.

6. **Target variable (readmitted) shows unique = 3** — confirming the three
   expected classes: NO, <30, >30. No unexpected values present.

7. **diag_1 shows 717 unique values** — ICD-9 diagnosis codes as expected.
   The sample value 250.83 is a diabetes-related code, consistent with this
   being a diabetes patient dataset.

**Next Step:** Sentinel value detection — quantify the true extent of
missing data hidden behind "?" strings before proceeding to cleaning.

In [9]:
# sentinel_profiling.py

def detect_sentinels(df, sentinel):
    counts = (df == sentinel).sum()
    pct    = (counts / len(df) * 100).round(2)
    report = pd.DataFrame({
        "sentinel_count": counts,
        "sentinel_pct"  : pct
    })
    report = (report[report["sentinel_count"] > 0]
              .sort_values("sentinel_pct", ascending=False))
    print(f"\n--- Sentinel '{sentinel}' Detection ---")
    print(report.to_string())
    return report

sentinel_report = detect_sentinels(df, CONFIG["missing_sentinel"])
sentinel_report.to_csv("profile_sentinel_report.csv")


--- Sentinel '?' Detection ---
                   sentinel_count  sentinel_pct
weight                      98569         96.86
medical_specialty           49949         49.08
payer_code                  40256         39.56
race                         2273          2.23
diag_3                       1423          1.40
diag_2                        358          0.35
diag_1                         21          0.02


In [10]:
# Checking what is actually inside the weight column
# weight is ~97% missing so should be full of "?"

print("--- weight column investigation ---")
print(f"dtype                : {df['weight'].dtype}")
print(f"Total rows           : {len(df)}")
print(f"Value counts (top 5) :")
print(df['weight'].value_counts(dropna=False).head())
print(f"\nDirect comparison df['weight'] == '?' :")
print((df['weight'] == '?').sum())
print(f"\nChecking for '?' using str.contains :")
print(df['weight'].astype(str).str.contains(r'^\?$', regex=True).sum())

--- weight column investigation ---
dtype                : str
Total rows           : 101766
Value counts (top 5) :
weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
Name: count, dtype: int64

Direct comparison df['weight'] == '?' :
98569

Checking for '?' using str.contains :
98569


In [11]:
# sentinal value detection
def detect_sentinels(df, sentinel):
    """
    Detects sentinel values across all string columns.
    Uses include=["object", "str"] for pandas 4 compatibility.
    """
    results = []

    string_cols = df.select_dtypes(include=["object", "str"]).columns

    for col in string_cols:
        count = (df[col] == sentinel).sum()
        pct   = round(count / len(df) * 100, 2)
        if count > 0:
            results.append({
                "column"        : col,
                "sentinel_count": count,
                "sentinel_pct"  : pct
            })

    if results:
        report = (pd.DataFrame(results)
                  .sort_values("sentinel_pct", ascending=False)
                  .reset_index(drop=True))
        print(f"\n--- Sentinel '{sentinel}' Detection ---")
        print(report.to_string(index=False))
        report.to_csv("outputs/profile_02_sentinel_report.csv", index=False)
        return report
    else:
        print(f"No sentinel '{sentinel}' values found in any column.")
        return pd.DataFrame()

sentinel_report = detect_sentinels(df, CONFIG["missing_sentinel"])


--- Sentinel '?' Detection ---
           column  sentinel_count  sentinel_pct
           weight           98569         96.86
medical_specialty           49949         49.08
       payer_code           40256         39.56
             race            2273          2.23
           diag_3            1423          1.40
           diag_2             358          0.35
           diag_1              21          0.02


##  Sentinel Value Detection

**Purpose:**
This dataset encodes missing values as the string "?" instead of NULL.
Until replaced, pandas treats them as valid strings — null counts remain
at zero and every aggregation produces misleading results. This step
reveals the true missingness picture before any cleaning begins.

**Key Observations:**

1. **weight** is the most affected column — 98,569 rows (96.86%) contain
   "?" confirming it is too sparse to use analytically. A weight_available
   flag will be created instead.

2. **medical_specialty** (~49%) and **payer_code** (~40%) have high sentinel
   rates — both will be filled with "Unknown" in the cleaning phase rather
   than imputed, as guessing specialty or payer information would corrupt
   the analysis.

3. **race** (~2%) has a small proportion of sentinels — filled with
   "Unknown". Demographics are never imputed.

4. **diag_2 and diag_3** have low sentinel rates — secondary and tertiary
   diagnoses are legitimately absent for simpler cases. These will be
   filled with "None".

**Why the initial function returned empty:**
The original function applied df == "?" across the entire DataFrame at once.
SQLAlchemy mixed-type object columns caused this comparison to silently
return 0 for affected columns. The fix iterates column by column across
object dtype columns only, which correctly identifies all sentinel values.

**Next Step:** Replace all "?" sentinel values with numpy NaN so pandas
correctly handles missingness in all downstream cleaning steps.

In [12]:
import pandas as pd
from config import CONFIG

def profile_cardinality(df, cols, top_n=10):
    results = []
    for col in cols:
        # Check if column exists in df to prevent KeyError
        if col not in df.columns:
            continue
            
        vc = df[col].value_counts(dropna=False).head(top_n)
        
        # 💡 Squelched the messy prints for Jupyter notebook cleanliness. 
        # The data is still captured perfectly in the background!
        
        for val, cnt in vc.items():
            results.append({
                "column": col,
                "value" : str(val),
                "count" : cnt,
                "pct"   : round(cnt / len(df) * 100, 2)
            })
    return pd.DataFrame(results)

# Assemble columns from your configuration
profile_cols = (
    CONFIG["categorical_cols"] +
    CONFIG["binary_cols"]      +
    CONFIG["medication_cols"]  +
    [CONFIG["target_col"]]
)

# Run the profile
cardinality_report = profile_cardinality(df, profile_cols)

# Save the full comprehensive background report
cardinality_report.to_csv("profile_cardinality_report.csv", index=False)

# ✨ Jupyter Special: Display just the high-cardinality trouble spots cleanly
# This filters the table to show you columns where things are fragmented
cardinality_report.head(20)

,column,value,count,pct
0,race,Caucasian,76099,74.78
1,race,AfricanAmerican,19210,18.88
2,race,?,2273,2.23
3,race,Hispanic,2037,2.00
4,race,Other,1506,1.48
5,race,Asian,641,0.63
6,gender,Female,54708,53.76
7,gender,Male,47055,46.24
8,gender,Unknown/Invalid,3,0.00
9,age,[70-80),26068,25.62


##  Cardinality and Value Distribution Profile

**Purpose:**
Inspect unique values and their frequencies per categorical column before
deciding a cleaning strategy. This step confirms expected values, reveals
unexpected entries, and surfaces any controlled vocabulary violations while
the data is still raw.

**Key Observations:**

**race (6 unique values)**
- Caucasian dominates at 74.78% — dataset is not demographically balanced
- AfricanAmerican is the second largest group at 18.88%
- 2,273 rows (2.23%) contain "?" — confirmed sentinel values requiring
  replacement with "Unknown" in the cleaning phase
- Demographics are never imputed — "Unknown" is the only appropriate fill

**gender (3 unique values)**
- Female 53.76% and Male 46.24% — reasonably balanced split
- 3 rows contain "Unknown/Invalid" — these are clinically unusable
  and will be flagged as gender_invalid_flag = 1 and excluded from
  all readmission rate calculations
- Note: only 3 rows affected — negligible impact on analysis

**age (10 unique values)**
- All 10 expected age bands present — no unexpected values
- Dataset is heavily skewed toward older patients as expected for a
  diabetes inpatient population:
    - [70-80) is the largest group at 25.62%
    - [60-70) second at 22.09%
    - Together [50-60) through [80-90) account for ~81% of encounters
- Very few young patients — [0-10) is only 0.16% (161 encounters)
- This age distribution is clinically consistent with Type 2 diabetes
  being predominantly a condition of older adults
- Age bands will remain as strings for grouping analysis
- Numeric midpoints will be extracted as age_midpoint feature for
  correlation analysis

**weight (10 unique values but 96.86% are "?")**
- 98,569 out of 101,766 rows are "?" — confirmed too sparse to use
  analytically
- Only 3,197 rows have actual weight values
- Strategy: add weight_available flag (1 = weight recorded, 0 = missing)
  and retain the column for audit trail — do not impute or drop

**Next Step:** Numeric distribution profile — summary statistics on all
numeric columns before cleaning.

In [13]:
# numeric_profiling.py
import pandas as pd
import numpy as np
from config import CONFIG

def profile_numerics(df, cols):
    stats = df[cols].agg([
        "count", "min", "max", "mean", "median", "std",
        lambda x: x.quantile(0.25),
        lambda x: x.quantile(0.75),
        "skew"
    ]).T
    stats.columns = [
        "count", "min", "max", "mean", "median", "std", "Q1", "Q3", "skew"
    ]
    stats["IQR"]   = stats["Q3"] - stats["Q1"]
    stats["range"] = stats["max"] - stats["min"]
    print(stats.round(2).to_string())
    return stats

numeric_stats_raw = profile_numerics(df, CONFIG["numeric_cols"])
numeric_stats_raw.to_csv("profile_numeric_raw.csv")

                       count  min    max   mean  median    std    Q1    Q3   skew   IQR  range
time_in_hospital    101766.0  1.0   14.0   4.40     4.0   2.99   2.0   6.0   1.13   4.0   13.0
num_lab_procedures  101766.0  1.0  132.0  43.10    44.0  19.67  31.0  57.0  -0.24  26.0  131.0
num_procedures      101766.0  0.0    6.0   1.34     1.0   1.71   0.0   2.0   1.32   2.0    6.0
num_medications     101766.0  1.0   81.0  16.02    15.0   8.13  10.0  20.0   1.33  10.0   80.0
number_outpatient   101766.0  0.0   42.0   0.37     0.0   1.27   0.0   0.0   8.83   0.0   42.0
number_emergency    101766.0  0.0   76.0   0.20     0.0   0.93   0.0   0.0  22.86   0.0   76.0
number_inpatient    101766.0  0.0   21.0   0.64     0.0   1.26   0.0   1.0   3.61   1.0   21.0
number_diagnoses    101766.0  1.0   16.0   7.42     8.0   1.93   6.0   9.0  -0.88   3.0   15.0


##  Numeric Distribution Profile

**Purpose:**
Generate extended summary statistics on all numeric columns before any
transformation. This establishes the pre-cleaning baseline for outlier
detection and provides the first clinical picture of patient complexity
and resource utilization across the 101,766 encounters.

**Key Observations:**

**time_in_hospital (Length of Stay)**
- Range: 1 to 14 days — note the dataset caps at 14 days maximum
- Mean: 4.40 days | Median: 4.0 days — closely aligned, suggesting
  a moderately symmetric core distribution
- Skew: 1.13 — right-skewed, meaning a minority of patients have
  significantly longer stays pulling the mean upward
- Most patients (IQR: 2 to 6 days) are discharged within a week
- Clinical interpretation: short stays may indicate early discharge
  which could contribute to readmission risk

**num_lab_procedures**
- Range: 1 to 132 — very wide range indicating high variability
  in clinical complexity across patients
- Mean: 43.10 | Median: 44.0 — nearly identical, suggesting a
  relatively normal distribution (confirmed by skew: -0.24)
- IQR: 31 to 57 — most patients receive between 31 and 57 lab tests
- This is the least skewed numeric column in the dataset

**num_procedures**
- Range: 0 to 6 — narrow range
- Mean: 1.34 | Median: 1.0 — most patients have 0 to 2 procedures
- Skew: 1.32 — right-skewed, most patients have few procedures
  with a small number having the maximum of 6
- Q1 = 0 means at least 25% of patients had no procedures at all

**num_medications**
- Range: 1 to 81 — extremely wide range
- Mean: 16.02 | Median: 15.0
- IQR: 10 to 20 — polypharmacy threshold of >= 10 medications
  captures the upper 75% of patients, making it a meaningful
  clinical segment for readmission analysis
- Skew: 1.33 — right-skewed with some patients on very high
  medication counts up to 81

**number_outpatient**
- Range: 0 to 42 | Mean: 0.37 | Median: 0.0
- Q1 = Q3 = 0 — at least 75% of patients had zero outpatient
  visits in the prior year
- Skew: 8.83 — extremely right-skewed, the most skewed column
  after number_emergency
- The max of 42 visits represents a high-utilization outlier

**number_emergency**
- Range: 0 to 76 | Mean: 0.20 | Median: 0.0
- Q1 = Q3 = 0 — at least 75% of patients had zero prior emergency
  visits
- Skew: 22.86 — the most extremely skewed column in the dataset
- Max of 76 emergency visits is a significant outlier — a patient
  with 76 prior emergency visits in one year represents extreme
  healthcare utilization
- Mann-Whitney U test (not a t-test) must be used for statistical
  comparisons on this column due to severe non-normality

**number_inpatient**
- Range: 0 to 21 | Mean: 0.64 | Median: 0.0
- Q1 = 0, Q3 = 1 — most patients had zero or one prior inpatient
  visit
- Skew: 3.61 — heavily right-skewed
- Despite the low median, prior inpatient visits are expected to
  be a strong predictor of readmission risk — this will be
  confirmed in the EDA and KPI phases

**number_diagnoses**
- Range: 1 to 16 | Mean: 7.42 | Median: 8.0
- Skew: -0.88 — the only column with notable negative skew,
  meaning most patients have a relatively high number of diagnoses
  with fewer patients having very low counts
- IQR: 6 to 9 — patients typically present with 6 to 9 diagnoses,
  indicating high clinical complexity consistent with a diabetic
  inpatient population

**Skewness Summary — Implications for Statistical Testing**

| Column | Skew | Distribution | Test to Use |
|---|---|---|---|
| time_in_hospital | 1.13 | Moderately right-skewed | Mann-Whitney U |
| num_lab_procedures | -0.24 | Approximately normal | Either test acceptable |
| num_procedures | 1.32 | Right-skewed | Mann-Whitney U |
| num_medications | 1.33 | Right-skewed | Mann-Whitney U |
| number_outpatient | 8.83 | Severely right-skewed | Mann-Whitney U |
| number_emergency | 22.86 | Extremely right-skewed | Mann-Whitney U |
| number_inpatient | 3.61 | Heavily right-skewed | Mann-Whitney U |
| number_diagnoses | -0.88 | Moderately left-skewed | Mann-Whitney U |

All numeric columns except num_lab_procedures are non-normal.
Mann-Whitney U will be used for all numeric comparisons in the
statistical validation phase. This justifies not using a standard
t-test, which assumes normality.

**Next Step:** Class imbalance documentation — examine the distribution
of the target variable (readmitted) before proceeding to cleaning.

In [14]:
# Define class imbalance documentation function
def document_class_imbalance(df, target_col):
    counts = df[target_col].value_counts()
    pct    = df[target_col].value_counts(normalize=True).mul(100).round(2)

    report = pd.DataFrame({
        "count"     : counts,
        "percentage": pct
    })
    report.index.name = target_col
    return report

In [15]:
# Run class imbalance check
imbalance_report = document_class_imbalance(df, CONFIG["target_col"])

print("--- Target Variable Class Distribution ---")
print(imbalance_report.to_string())

# Imbalance ratio
majority = imbalance_report["percentage"].max()
minority = imbalance_report["percentage"].min()
ratio    = majority / minority

print(f"\nMajority class : {imbalance_report['percentage'].idxmax()} ({majority}%)")
print(f"Minority class : {imbalance_report['percentage'].idxmin()} ({minority}%)")
print(f"Imbalance ratio: {ratio:.1f}:1")

--- Target Variable Class Distribution ---
            count  percentage
readmitted                   
NO          54864       53.91
>30         35545       34.93
<30         11357       11.16

Majority class : NO (53.91%)
Minority class : <30 (11.16%)
Imbalance ratio: 4.8:1


##  Class Imbalance Documentation

**Purpose:**
Document the distribution of the target variable (readmitted) before
any cleaning or analysis. The readmitted column has three classes:
- NO  : patient was not readmitted
- >30 : patient was readmitted after 30 days
- <30 : patient was readmitted within 30 days (the primary KPI target)

This must be documented because:
1. Every readmission rate KPI is a proportion of an imbalanced population
2. The minority class (<30) drives the entire analysis
3. Any future predictive modeling extension must address this imbalance
4. Documenting it shows analytical maturity that surface-level projects skip

## Class Imbalance Documentation — Findings

**Target Variable Distribution:**

| Class | Count | Percentage | Meaning |
|---|---|---|---|
| NO  | 54,864 | 53.91% | Patient was not readmitted after discharge |
| >30 | 35,545 | 34.93% | Patient was readmitted but after 30 days |
| <30 | 11,357 | 11.16% | Patient was readmitted within 30 days |

**Imbalance Ratio: 4.8:1**
(majority class NO vs minority class <30)

**Key Findings:**

1. **The dataset is imbalanced** — the primary target of interest (<30 day
   readmission) represents only 11.16% of all encounters. For every patient
   readmitted within 30 days, there are approximately 4.8 patients who were
   not readmitted at all.

2. **>30 is a substantial middle group** — 34.93% of patients were readmitted
   but outside the 30-day window. This group is clinically significant but
   is not the primary KPI target. It will be tracked separately in the
   SQL analysis as led_to_any_readmission.

3. **The 30-day readmission rate baseline is 11.16%** — this is the headline
   KPI number. Every segmentation in the EDA and KPI views will be compared
   against this overall rate to identify which groups are above or below it.
   A diagnosis category showing 18% readmission rate is nearly double the
   baseline — that is a clinically meaningful finding.

4. **Implication for KPI interpretation** — all readmission rates computed
   in SQL are proportions of this imbalanced population. When a specialty
   shows a 15% readmission rate, it means 15 out of every 100 encounters
   in that specialty led to a readmission within 30 days — always interpreted
   against the 11.16% overall baseline.

5. **Implication for future modeling** — if this project is extended to
   predictive modeling, the 4.8:1 imbalance means a naive model could
   achieve 53.91% accuracy simply by predicting NO for every patient.
   Techniques such as SMOTE oversampling or class weighting would be
   required. This is outside the scope of the current descriptive
   analytics project but is documented here for completeness.

**This completes the data profiling phase.**
All 5 profiling outputs have been saved to the outputs/ folder:
- profile_01_raw_structure.csv
- profile_02_sentinel_report.csv
- profile_03_cardinality.csv
- profile_04_numeric_raw.csv
- profile_05_class_imbalance.csv

**Next Phase:** Data cleaning — 02_data_cleaning.ipynb
Starting with sentinel value replacement, the cleaning phase will
address all issues identified during profiling.

In [16]:
# Save to outputs folder
import os
os.makedirs("outputs", exist_ok=True)

imbalance_report.to_csv("outputs/profile_05_class_imbalance.csv")
print("Class imbalance report saved to outputs/profile_05_class_imbalance.csv")

Class imbalance report saved to outputs/profile_05_class_imbalance.csv
